In [19]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import BaseMessage, HumanMessage
from dotenv import load_dotenv
from typing import TypedDict, Literal, Annotated, List


In [20]:
load_dotenv()  # Load environment variables from .env file

True

In [21]:
class ChatState(TypedDict):
    
    messages: Annotated[List[BaseMessage], add_messages]

In [28]:
llm = ChatAnthropic(model="claude-sonnet-5")

def chat_node(state: ChatState) -> ChatState:
    # Get the messages from the state
    messages = state["messages"]

    # Call the LLM with the messages
    response = llm.invoke(messages)

    # Return the updated state
    return {"messages": [response]}

In [29]:
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile()

In [30]:
initial_state = {
    "messages": [HumanMessage(content="What is the capital of France?")]
}
final_state = chatbot.invoke(initial_state)

In [35]:
final_state['messages'][1].content

"The capital of France is **Paris**.\n\nIt's the country's largest city and serves as its political, economic, and cultural center. Paris is known for landmarks such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral."